In [5]:
import cv2
import numpy as np
import random
import json
from pathlib import Path
from tqdm import tqdm

In [7]:
random.seed(42)
np.random.seed(42)

def degrade_saturation(img, s):
    """img: float32 HWC [0,1]. Применяется ПЕРВОЙ (обратный порядок к коррекции)."""
    lum = (0.299 * img[..., 0] + 0.587 * img[..., 1] + 0.114 * img[..., 2])[..., None]
    return np.clip(lum + (img - lum) * s, 0, 1)

def degrade_brightness_contrast(img, b, c):
    """Применяется ВТОРОЙ."""
    return np.clip((img - 0.5) * c + 0.5 + b, 0, 1)

def degrade_image(img, b, c, s):
    out = degrade_saturation(img, s)
    out = degrade_brightness_contrast(out, b, c)
    return out

def sample_degrade_params(severity="mixed"):
    if severity == "mixed":
        severity = np.random.choice(["mild", "severe"], p=[0.35, 0.65])

    if severity == "mild":
        b = np.random.uniform(-0.3, 0.3)
        c = np.exp(np.random.uniform(np.log(0.45), np.log(2.0)))
        s = np.exp(np.random.uniform(np.log(0.3), np.log(2.2)))
    else:  # severe
        b = np.random.uniform(-0.55, 0.55)
        c = np.exp(np.random.uniform(np.log(0.2), np.log(3.2)))
        s = np.exp(np.random.uniform(np.log(0.05), np.log(3.5)))

    return b, c, s

def invert_params(b_deg, c_deg, s_deg):
    c_corr = 1.0 / c_deg
    b_corr = -b_deg / c_deg
    s_corr = 1.0 / s_deg
    return b_corr, c_corr, s_corr

def process_files(files, out_dir, n_variants_per_image):
    out_dir = Path(out_dir)
    (out_dir / "degraded").mkdir(parents=True, exist_ok=True)

    targets = {}
    for f in tqdm(files, desc=out_dir.name):
        img = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

        for v in range(n_variants_per_image):
            b_deg, c_deg, s_deg = sample_degrade_params()
            degraded = degrade_image(img, b_deg, c_deg, s_deg)

            out_name = f"{f.stem}_v{v}.jpg"
            out_path = out_dir / "degraded" / out_name
            cv2.imwrite(str(out_path), cv2.cvtColor((degraded * 255).astype(np.uint8), cv2.COLOR_RGB2BGR))

            b_corr, c_corr, s_corr = invert_params(b_deg, c_deg, s_deg)
            targets[out_name] = {"brightness": b_corr, "contrast": c_corr, "saturation": s_corr}

    with open(out_dir / "targets.json", "w") as fp:
        json.dump(targets, fp, indent=2)

    print(f"{out_dir}: готово {len(targets)} пар изображение/параметры")

if __name__ == "__main__":
    good_dir = Path("data")
    files = sorted(good_dir.glob("*.png"))
    random.shuffle(files)

    n = len(files)
    train_files = files[:int(n * 0.8)]
    val_files   = files[int(n * 0.8):int(n * 0.9)]
    test_files  = files[int(n * 0.9):]

    print(f"train={len(train_files)} val={len(val_files)} test={len(test_files)}")

    # больше вариантов на train, т.к. DIV2K меньше по объёму, чем FiveK
    process_files(train_files, "data/synthetic/train", n_variants_per_image=7)
    process_files(val_files,   "data/synthetic/val",   n_variants_per_image=3)
    process_files(test_files,  "data/synthetic/test",  n_variants_per_image=3)

train=720 val=90 test=90


train: 100%|██████████| 720/720 [30:09<00:00,  2.51s/it]


data\synthetic\train: готово 5040 пар изображение/параметры


val: 100%|██████████| 90/90 [01:50<00:00,  1.23s/it]


data\synthetic\val: готово 270 пар изображение/параметры


test: 100%|██████████| 90/90 [02:44<00:00,  1.82s/it]

data\synthetic\test: готово 270 пар изображение/параметры
